字符序列："ababc"
相邻转移统计：a->b, b->a, a->b, b->c
以字符 'b' 为条件：总出现次数 count(b) = 2
词汇表大小 |V| = 3 （即 {'a','b','c'}）
拉普拉斯平滑（加1平滑）公式：P(xt | x_{t-1}) = [count(x_{t-1}, xt) + 1] / [count(x_{t-1}) + |V|]

P('a' | 'b') = (1 + 1) / (2 + 3) = 2 / 5 = 0.4

P('c' | 'b') = (1 + 1) / (2 + 3) = 2 / 5 = 0.4

（隐含的 P('b' | 'b') = (0 + 1) / (2 + 3) = 1 / 5 = 0.2，总和为 1.0）

In [7]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写，去除非字母和空格
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower()
    # 2. 按空格分词
    tokens = text.split()
    # 3. 构建词汇表（按频率排序）
    counter = Counter(tokens)
    vocab = {word: idx for idx, (word, _) in enumerate(sorted(counter.items(), key=lambda x: -x[1]))}
    # 4. 滑动窗口
    features, labels = [], []
    for i in range(len(tokens) - n + 1):
        features.append(tokens[i:i+n])
        labels.append(tokens[i+n] if i+n < len(tokens) else None)
    return vocab, (features, labels)

# ========== 验证部分 ==========
if __name__ == "__main__":
    vocab, (features, labels) = preprocess_text("The time machine", 2)
    print("词汇表:", vocab)
    print("特征序列:", features)
    print("标签序列:", labels)

# 预期输出：
# 词汇表: {'the': 0, 'time': 1, 'machine': 2} (频率相同按原序)
# 特征序列: [['the', 'time'], ['time', 'machine']]
# 标签序列: ['machine', None]

词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征序列: [['the', 'time'], ['time', 'machine']]
标签序列: ['machine', None]


线性RNN定义：
h_t = W_hh * h_{t-1} + W_hx * x_t
o_t = W_oh * h_t
平方损失：L = 1/2 * sum_{t=1}^{T} (o_t - y_t)^2

设 e_t = o_t - y_t，定义损失对隐藏状态的梯度 delta_t = dL / dh_t。
从最后一步往前推：
delta_T = (W_oh)^T * e_T
对于 t < T：delta_t = (W_oh)^T * e_t + (W_hh)^T * delta_{t+1}

最终权重梯度（展开到所有时间步）：
dL / dW_hh = sum_{t=1}^{T} [ delta_t * (h_{t-1})^T ]

梯度爆炸或消失的条件（取决于矩阵 W_hh 的谱半径 rho）：
若 rho(W_hh) > 1，则随着时间步增加，梯度项 (W_hh)^(T-t) 会指数增长，导致梯度爆炸。
若 rho(W_hh) < 1，则梯度项会指数衰减趋近于 0，导致梯度消失。



In [8]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    z = x_t @ W_hx.T + h_prev @ W_hh.T + b_h
    h_t = np.tanh(z)
    return h_t, z

def rnn_cell_backward(dh_next, x_t, h_prev, h_t, W_hx, W_hh):
    dz = dh_next * (1 - h_t ** 2)
    dx_t = dz @ W_hx
    dh_prev = dz @ W_hh
    dW_hx = dz.T @ x_t
    dW_hh = dz.T @ h_prev
    db_h = np.sum(dz, axis=0)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# ========== 验证部分 ==========
if __name__ == "__main__":
    np.random.seed(42)
    batch, in_dim, hid_dim = 2, 3, 4
    x_t = np.random.randn(batch, in_dim)
    h_prev = np.random.randn(batch, hid_dim)
    W_hx = np.random.randn(hid_dim, in_dim)
    W_hh = np.random.randn(hid_dim, hid_dim)
    b_h = np.random.randn(hid_dim)
    dh_next = np.random.randn(batch, hid_dim)

    h_t, z = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
    dx, dh_prev, dWx, dWh, db = rnn_cell_backward(dh_next, x_t, h_prev, h_t, W_hx, W_hh)

    print("前向 h_t shape:", h_t.shape)          # (2, 4)
    print("反向 dx_t shape:", dx.shape)          # (2, 3)
    print("反向 dh_prev shape:", dh_prev.shape)  # (2, 4)
    print("反向 dW_hx shape:", dWx.shape)        # (4, 3)
    print("反向 dW_hh shape:", dWh.shape)        # (4, 4)
    print("反向 db_h shape:", db.shape)          # (4,)

前向 h_t shape: (2, 4)
反向 dx_t shape: (2, 3)
反向 dh_prev shape: (2, 4)
反向 dW_hx shape: (4, 3)
反向 dW_hh shape: (4, 4)
反向 db_h shape: (4,)


深度双向RNN参数计算（L层，每层H个隐藏单元，输入维度D，输出维度O）：
第1层（输入来自外部D维）：每个方向参数为 H*(D+H+1)，双向即 2*H*(D+H+1)。
第2层至第L层（输入来自上一层双向拼接，维度为 2*H）：每个方向参数为 H*(2H+H+1) = H*(3H+1)，双向即 2H(3H+1)。共 L-1 层。
输出层（最后一层双向拼接 2H 映射到 O）：参数为 O(2H+1)。

总参数总数表达式：
Total = 2H(D+H+1) + (L-1)*2*H*(3H+1) + O*(2H+1)

In [9]:
import torch
import torch.nn as nn

def bidirectional_rnn_encoder(X, input_dim, hidden_dim, num_layers=1):
    rnn = nn.RNN(
        input_size=input_dim,
        hidden_size=hidden_dim,
        num_layers=num_layers,
        bidirectional=True,
        batch_first=False
    )
    output, _ = rnn(X)          # output: (seq_len, batch, 2*hidden_dim)
    final = output[-1]          # (batch, 2*hidden_dim)
    return output, final

# ========== 验证部分 ==========
if __name__ == "__main__":
    seq_len, batch, dim = 5, 3, 8
    hidden = 4
    X = torch.randn(seq_len, batch, dim)
    output, final = bidirectional_rnn_encoder(X, dim, hidden, num_layers=1)
    print("输出张量 output 形状:", output.shape)   # torch.Size([5, 3, 8])
    print("最终隐藏状态 final 形状:", final.shape) # torch.Size([3, 8])

输出张量 output 形状: torch.Size([5, 3, 8])
最终隐藏状态 final 形状: torch.Size([3, 8])


Skip-gram 负采样目标函数（最大化对数似然，等价于最小化负对数似然）：
给定中心词 w_c（向量 v_c），正样本上下文词 w_o（向量 u_o），负样本词 n_k（向量 u_{n_k}），采样 K 个负样本。

损失函数（需最小化）为：
L = - log( sigma( u_o^T * v_c ) ) - sum_{k=1}^{K} log( sigma( - u_{n_k}^T * v_c ) )
其中 sigma(x) = 1 / (1 + exp(-x))。

负样本从噪声分布中采样，常用的噪声分布为：
P_n(w) = [ count(w)^{3/4} ] / [ sum_{w' in Vocab} count(w')^{3/4} ]
该分布通过 3/4 次幂平滑了高频词和低频词的采样概率。

In [10]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, target_indices, W, W_out):
    # context_indices: (batch, context_size)
    emb = W[context_indices]                 # (batch, context_size, d)
    h = emb.mean(dim=1)                      # (batch, d)
    logits = h @ W_out                       # (batch, vocab_size)
    loss = F.cross_entropy(logits, target_indices)
    return loss

# ========== 验证部分 ==========
if __name__ == "__main__":
    torch.manual_seed(1)
    V, d, ctx_size, batch = 10, 4, 3, 2
    W = torch.randn(V, d, requires_grad=False)
    W_out = torch.randn(d, V, requires_grad=False)
    context = torch.randint(0, V, (batch, ctx_size))
    target = torch.randint(0, V, (batch,))

    loss = cbow_forward(context, target, W, W_out)
    print("CBOW 交叉熵损失值:", loss.item())   # 例如: 2.3517

CBOW 交叉熵损失值: 3.7279367446899414


给定 Q (2行4列)，K (3行4列)，V (3行5列)，dk = 4，所以 sqrt(dk) = 2。
计算过程分三步：

第一步：计算得分矩阵 S = (Q * K^T) / 2。
S 是一个 2行3列 的矩阵。
（具体数值需自行代入矩阵元素，此处只写步骤公式）
S[i][j] = ( Q的第i行 与 K的第j行 的点积 ) / 2

第二步：对得分矩阵 S 的每一行分别应用 Softmax 函数，得到注意力权重矩阵 A。
A[i][j] = exp(S[i][j]) / ( exp(S[i][1]) + exp(S[i][2]) + exp(S[i][3]) )
A 同样为 2行3列，且每一行之和为 1。

第三步：计算输出矩阵 O = A * V。
O 为 2行5列。
O 的第 i 行 = A[i][1]*V第1行 + A[i][2]*V第2行 + A[i][3]*V第3行。

In [11]:
import torch
import torch.nn.functional as F
import math

def multi_head_attention_forward(X, num_heads=2, d_model=4):
    d_k = d_model // num_heads
    seq_len, batch, _ = X.shape

    # 为了演示初始化（实际训练应用 nn.Parameter）
    W_q = torch.randn(d_model, d_model)
    W_k = torch.randn(d_model, d_model)
    W_v = torch.randn(d_model, d_model)
    W_o = torch.randn(d_model, d_model)

    Q = X @ W_q.T
    K = X @ W_k.T
    V = X @ W_v.T

    # 切分多头: (seq, batch, heads, d_k)
    Q = Q.view(seq_len, batch, num_heads, d_k).permute(1, 2, 0, 3).reshape(batch*num_heads, seq_len, d_k)
    K = K.view(seq_len, batch, num_heads, d_k).permute(1, 2, 0, 3).reshape(batch*num_heads, seq_len, d_k)
    V = V.view(seq_len, batch, num_heads, d_k).permute(1, 2, 0, 3).reshape(batch*num_heads, seq_len, d_k)

    # 缩放点积
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    attn = F.softmax(scores, dim=-1)
    out = torch.bmm(attn, V)  # (batch*heads, seq_len, d_k)

    # 合并多头
    out = out.view(batch, num_heads, seq_len, d_k).permute(2, 0, 1, 3).reshape(seq_len, batch, d_model)
    out = out @ W_o.T
    return out

# ========== 验证部分 ==========
if __name__ == "__main__":
    torch.manual_seed(123)
    seq, batch, model_dim = 6, 2, 4
    X = torch.randn(seq, batch, model_dim)
    out = multi_head_attention_forward(X, num_heads=2, d_model=4)
    print("多头注意力输出形状:", out.shape)  # torch.Size([6, 2, 4])

多头注意力输出形状: torch.Size([6, 2, 4])
